In [1]:
import os
if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 0 # Use "" to use the CPU
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'


import sys
sys.path.append('../')
sys.path.append('/content/thanh/')
import sionna

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)
# Avoid warnings from TensorFlow
tf.get_logger().setLevel('ERROR')

sionna.config.seed = 42 # Set seed for reproducible random number generation

# Load the required Sionna components
from sionna.nr import PUSCHConfig, PUSCHTransmitter, PUSCHReceiver, CarrierConfig, PUSCHDMRSConfig,\
                        TBConfig, PUSCHPilotPattern, TBEncoder, PUSCHPrecoder, LayerMapper, LayerDemapper, check_pusch_configs,\
                        TBDecoder, PUSCHLSChannelEstimator
from sionna.nr.utils import generate_prng_seq
from sionna.channel import AWGN, RayleighBlockFading, OFDMChannel, TimeChannel, time_lag_discrete_time_channel
from sionna.channel.utils import *
from sionna.channel.tr38901 import Antenna, AntennaArray, UMi, UMa, RMa, TDL, CDL
from sionna.channel import gen_single_sector_topology as gen_topology
from sionna.utils import compute_ber, ebnodb2no, sim_ber, BinarySource
from sionna.ofdm import KBestDetector, LinearDetector, MaximumLikelihoodDetector,\
        LSChannelEstimator, LMMSEEqualizer, RemoveNulledSubcarriers, ResourceGridDemapper,\
        ResourceGrid, ResourceGridMapper, OFDMModulator
from sionna.mimo import StreamManagement
from sionna.mapping import Mapper, Demapper

from sionna.nr.my_abc import *

In [2]:
from pathlib import Path
import struct
import numpy as np

In [1]:
def data_writer(data_path, data):
    """Writes complex int16 data into a binary file."""
    data = data.flatten()
    with open(data_path, 'wb') as file:
        for value in data:
            real_part = struct.pack('<h', int(value.real))
            imag_part = struct.pack('<h', int(value.imag))
            file.write(real_part + imag_part)

def data_reader(data_path, sysInfo, ueInfo):
    freq = []
    with open(data_path, 'rb') as file:
        binary_data = file.read()
        # Loop through the binary data in chunks of 2 bytes (since int16 is 2 bytes)
        for i in range(0, len(binary_data), 4):
            # Convert the chunk to an integer and append it to the list
            real = binary_data[i:i+2]
            imag = binary_data[i+2:i+4]
            #unpack the 2 bytes into a little-endian int16
            if len(real) == 2:
                real_part = struct.unpack('<h', real)[0]
                imag_part = struct.unpack('<h', imag)[0]
                freq.append(complex(real_part, imag_part))
    # return 
    # iqFreq = np.array(freq, dtype= np.complex64)
    # iqFreq = np.transpose(np.reshape(iqFreq, (sysInfo['BwpNRb']*12, 14, sysInfo['NRxAnt'])), [2,1,0])

    iqFreq = np.reshape(iqFreq, (sysInfo['NRxAnt'], 14, sysInfo['BwpNRb']*12))
    return iqFreq
    # iqFreq = tf.transpose(iqFreq, [2,1,0])
     
    # iqUe = iqFreq[ueInfo[0]['FirstPrb']*12:(ueInfo[0]['NPrb'] + ueInfo[0]['FirstPrb'])*12, :, :]

    # return iqUe, iqFreq
    

field_dir = '../Pusch_data/data_field/20_12_2024/dump_data20_12/dump_lan1_2dmrs'
data_path = f'{field_dir}/dump_pass_sfn_718_sf_4_freq.bin'
cfg_path = f'{field_dir}/dump_pass_sfn_718_sf_4_cfg.txt'


# Load configuration parameters from the file .txt
caseInfo, sysInfo, ueInfo, chcfg, auxInfo = config_parser(cfg_path)

# iqUe, iqFreq = data_reader(data_path, sysInfo, ueInfo)

freq = data_reader(data_path, sysInfo, ueInfo)

NameError: name 'config_parser' is not defined

In [2]:
data_writer(f'{field_dir}/dump_pass_sfn_718_sf_4_freq_rec.bin', freq)

NameError: name 'freq' is not defined

In [13]:
freq_rec = data_reader(f'{field_dir}/dump_pass_sfn_718_sf_4_freq_rec.bin', sysInfo, ueInfo)

In [14]:
(freq_rec - freq).var()

53200.16

In [15]:
sysCfg = SystemConfig(**sysInfo)
ueCfgs = [UeConfig(**ue) for ue in ueInfo.values()]
myCfg = MyConfig(sysCfg, ueCfgs)
puschCfg = MyPUSCHConfig(myCfg)
puschCfg.show()

Carrier Configuration
cyclic_prefix : normal
cyclic_prefix_length : 2.3437500000000002e-06
frame_duration : 0.01
frame_number : 0
kappa : 64.0
mu : 1
n_cell_id : 442
n_size_grid : 162
n_start_grid : 0
num_slots_per_frame : 20
num_slots_per_subframe : 2
num_symbols_per_slot : 14
slot_number : 4
sub_frame_duration : 0.001
subcarrier_spacing : 30
t_c : 5.086263020833334e-10
t_s : 3.2552083333333335e-08

PUSCH Configuration
My_Config : MyConfig(Sys=SystemConfig(NCellId=442, FrequencyRange=1, BandWidth=60, Numerology=1, CpType=0, NTxAnt=1, NRxAnt=8, BwpNRb=162, BwpRbOffset=0, harqProcFlag=0, nHarqProc=1, rvSeq=0), Ue=[UeConfig(TransformPrecoding=0, Rnti=30053, nId=442, CodeBookBased=0, DmrsPortSetIdx=[0], NLayers=1, NumDmrsCdmGroupsWithoutData=2, Tpmi=0, FirstSymb=0, NPuschSymbAll=14, RaType=1, FirstPrb=25, NPrb=137, FrequencyHoppingMode=0, McsTable=0, Mcs=8, ILbrm=0, nScId=0, NnScIdId=442, DmrsConfigurationType=0, DmrsDuration=1, DmrsAdditionalPosition=1, PuschMappingType=0, DmrsTypeAPosit

In [16]:
tv_dir ='../Pusch_data/test_vector/test-1234'
os.makedirs(tv_dir, exist_ok=True)
slots = [4]
setCfgReq(puschCfg=puschCfg, slots=slots, dir=tv_dir)
for n,i in enumerate(slots):
    setUlCfgReq(puschCfg=puschCfg, harqIdx=n%16, dir=tv_dir)
set_null(dir=tv_dir)


rxSigFreq = tf.Variable(tf.zeros([8, 20, 14, 1944], dtype=tf.complex64))
for n,i in enumerate(slots):
    rxSigFreq[:,i,:,:].assign(freq)
setRxData(rxSigFreq, dir=tv_dir)
setRef(None, dir=tv_dir)


Config Request saved to ../Pusch_data/test_vector/test-1234/cfgReq.cfg
UL Config Request Slot 4 saved to ../Pusch_data/test_vector/test-1234/slot4.cfg
Null Config saved to ../Pusch_data/test_vector/test-1234/null.cfg


In [ ]:
# def setCfgReq(puschCfg: MyPUSCHConfig, slots, dir):
#     bandwidth = puschCfg.My_Config.Sys.BandWidth
#     fft_size = 4096
#     filename = os.path.join(dir, 'cfgReq.cfg')
    
#     slot_config_template = "1,1,1,1,1,1,1,1,1,1,1,1,1,1"  # 14 ones
#     default_config_template = "2,2,2,2,2,2,2,2,2,2,2,2,2,2"  # 14 twos
    
#     xml_content = """<?xml version="1.0"?>

# <TestConfig>
# 	<numSlots>20</numSlots>
# """
#     for i in range(8):
#         xml_content += f"\t<uliq_car0_ant{i}>rx_ant_{i}.bin</uliq_car0_ant{i}>\n"
    
#     xml_content += f"""	<ul_ref_out>ref.txt</ul_ref_out>
# 	<start_frame_number>0</start_frame_number>
# 	<start_slot_number>0</start_slot_number>
# </TestConfig>

# <ConfigReq>
# 	<nCarrierIdx>0</nCarrierIdx>
# 	<nDMRSTypeAPos>3</nDMRSTypeAPos>
# 	<nPhyCellId>{puschCfg.carrier.n_cell_id}</nPhyCellId>
# 	<nDLBandwidth>{bandwidth}</nDLBandwidth>
# 	<nULBandwidth>{bandwidth}</nULBandwidth>
# 	<nDLFftSize>{fft_size}</nDLFftSize>
# 	<nULFftSize>{fft_size}</nULFftSize>
# 	<nNrOfTxAnt>8</nNrOfTxAnt>
# 	<nNrOfRxAnt>8</nNrOfRxAnt>
# 	<nCarrierAggregationLevel>0</nCarrierAggregationLevel>
# 	<nFrameDuplexType>1</nFrameDuplexType>
# 	<nSubcCommon>1</nSubcCommon>
# 	<nTddPeriod>20</nTddPeriod>
# """
    
#     for i in range(20):
#         config_value = slot_config_template if i in slots else default_config_template
#         xml_content += f"\t<sSlotConfig{i}>{config_value}</sSlotConfig{i}>\n"
    
#     xml_content += "\t<nCyclicPrefix>0</nCyclicPrefix>\n</ConfigReq>\n\n<RxConfig>\n"
    
#     for i in range(20):
#         slot_value = f"slot{i}.cfg" if i in slots else "null.cfg"
#         xml_content += f"\t<SlotNum{i}>{slot_value}</SlotNum{i}>\n"
    
#     xml_content += """</RxConfig>
# """
    
#     with open(filename, 'w') as f:
#         f.write(xml_content)
    
#     print(f"Config Request saved to {filename}")

# def setUlCfgReq(puschCfg: MyPUSCHConfig, harqIdx, dir):

#     filename = os.path.join(dir, f'slot{puschCfg.carrier.slot_number}.cfg')

#     # Define the XML structure dynamically
#     xml_content = f"""<?xml version="1.0"?>

# <Ul_Config_Req>

# 	<UlConfigReqL1L2Header>
# 		<nSFN>{puschCfg.carrier.frame_number}</nSFN>
# 		<nSlot>{puschCfg.carrier.slot_number}</nSlot>
# 		<nPDU>{1}</nPDU>
# 		<nGroup>{1}</nGroup>
# 		<nUlsch>{1}</nUlsch>
# 		<nUlcch>{0}</nUlcch>
# 		<nRachPresent>{0}</nRachPresent>
# 	</UlConfigReqL1L2Header>

# 	<UL_SCH_PDU0>
# 		<nRNTI>{puschCfg.n_rnti}</nRNTI>
# 		<nUEId>{0}</nUEId>
# 		<nBWPSize>{puschCfg.n_size_bwp}</nBWPSize>
# 		<nBWPStart>{puschCfg.n_start_bwp}</nBWPStart>
# 		<nSubcSpacing>{puschCfg.carrier.mu}</nSubcSpacing>
# 		<nCpType>{puschCfg.My_Config.Sys.CpType}</nCpType>
# 		<nULType>{0}</nULType>
# 		<nMcsTable>{puschCfg.tb.mcs_table - 1}</nMcsTable>
# 		<nMCS>{puschCfg.tb.mcs_index}</nMCS>
# 		<nTransPrecode>{puschCfg.My_Config.Ue[0].TransformPrecoding}</nTransPrecode>
# 		<nTransmissionScheme>{puschCfg.My_Config.Ue[0].CodeBookBased}</nTransmissionScheme>
# 		<nNrOfLayers>{puschCfg.num_layers}</nNrOfLayers>
# 		<nPortIndex0>{puschCfg.dmrs.dmrs_port_set[0]}</nPortIndex0>
# 		<nNid>{puschCfg.tb.n_id}</nNid>
# 		<nSCID>{puschCfg.dmrs.n_scid}</nSCID>
# 		<nNIDnSCID>{puschCfg.dmrs.n_id[0]}</nNIDnSCID>
# 		<nNrOfAntennaPorts>{puschCfg.My_Config.Sys.NRxAnt}</nNrOfAntennaPorts>
# 		<nVRBtoPRB>{0}</nVRBtoPRB>
# 		<nPMI>{puschCfg.My_Config.Ue[0].Tpmi}</nPMI>
# 		<nStartSymbolIndex>{puschCfg.symbol_allocation[0]}</nStartSymbolIndex>
# 		<nNrOfSymbols>{puschCfg.symbol_allocation[1]}</nNrOfSymbols>
# 		<nResourceAllocType>{1}</nResourceAllocType>
# 		<nRBStart>{puschCfg.first_resource_block}</nRBStart>
# 		<nRBSize>{puschCfg.num_resource_blocks}</nRBSize>
# 		<nTBSize>{(puschCfg.tb_size//8)}</nTBSize>
# 		<nRV>{puschCfg.My_Config.Sys.rvSeq}</nRV>
# 		<nHARQID>{harqIdx}</nHARQID>
# 		<nNDI>{1}</nNDI>
# 		<nMappingType>{puschCfg.My_Config.Ue[0].PuschMappingType}</nMappingType>
# 		<nDMRSConfigType>{puschCfg.My_Config.Ue[0].DmrsConfigurationType}</nDMRSConfigType>
# 		<nNrOfCDMs>{puschCfg.dmrs.num_cdm_groups_without_data}</nNrOfCDMs>
# 		<nNrOfDMRSSymbols>{puschCfg.dmrs.length}</nNrOfDMRSSymbols>
# 		<nDMRSAddPos>{puschCfg.dmrs.additional_position}</nDMRSAddPos>
# 		<nPTRSPresent>{puschCfg.My_Config.Ue[0].Ptrs}</nPTRSPresent>
# 		<nAck>{puschCfg.My_Config.Ue[0].OAck}</nAck>
# 		<nAlphaScaling>{puschCfg.My_Config.Ue[0].ScalingFactor}</nAlphaScaling>
# 		<nBetaOffsetACKIndex>{puschCfg.My_Config.Ue[0].IHarqAckOffset}</nBetaOffsetACKIndex>
# 		<nCsiPart1>{puschCfg.My_Config.Ue[0].OCsi1}</nCsiPart1>
# 		<nBetaOffsetCsiPart1Index>{puschCfg.My_Config.Ue[0].ICsi1Offset}</nBetaOffsetCsiPart1Index>
# 		<nCsiPart2>{puschCfg.My_Config.Ue[0].OCsi2}</nCsiPart2>
# 		<nBetaOffsetCsiPart2Index>{puschCfg.My_Config.Ue[0].ICsi2Offset}</nBetaOffsetCsiPart2Index>
# 		<nTpPi2BPSK>{puschCfg.My_Config.Ue[0].TpPi2Bpsk}</nTpPi2BPSK>
# 		<nTPPuschID>{puschCfg.My_Config.Ue[0].NRsId}</nTPPuschID>
# 		<nRxRUIdx0>{0}</nRxRUIdx0>
# 		<nRxRUIdx1>{1}</nRxRUIdx1>
# 		<nRxRUIdx2>{2}</nRxRUIdx2>
# 		<nRxRUIdx3>{3}</nRxRUIdx3>
# 		<nRxRUIdx4>{4}</nRxRUIdx4>
# 		<nRxRUIdx5>{5}</nRxRUIdx5>
# 		<nRxRUIdx6>{6}</nRxRUIdx6>
# 		<nRxRUIdx7>{7}</nRxRUIdx7>
# 	</UL_SCH_PDU0>

# 	<PUSCH_GROUP_INFO0>
# 		<nUE>{1}</nUE>
# 		<nPduIdx0>{0}</nPduIdx0>
# 	</PUSCH_GROUP_INFO0>

# </Ul_Config_Req>
# """
#     with open(filename, 'w') as file:
#         file.write(xml_content)
#     print(f"UL Config Request Slot {puschCfg.carrier.slot_number} saved to {filename}")

# def set_null(dir):
#     filename = os.path.join(dir, 'null.cfg')

#     xml_content = f"""<?xml version="1.0"?>

# <Ul_Config_Req>
# 	<UlConfigReqL1L2Header>
# 		<nSFN>0</nSFN>
# 		<nSlot>0</nSlot>
# 		<nPDU>0</nPDU>
# 		<nGroup>0</nGroup>
# 		<nUlsch>0</nUlsch>
# 		<nUlcch>0</nUlcch>
# 		<nRachPresent>0</nRachPresent>
# 	</UlConfigReqL1L2Header>
# </Ul_Config_Req>
# """
#     with open(filename, 'w') as file:
#         file.write(xml_content)
#     print(f"Null Config saved to {filename}")
    
# def setRxData(rxSigFreq, dir):
#     # rxSigFreq = tf.reshape(rxSigFreq, -1)
#     rxSigFreq = tf.stack((tf.math.real(rxSigFreq), tf.math.imag(rxSigFreq)), axis=-1)
    
#     rxSigFreq = rxSigFreq/tf.math.reduce_max(tf.abs(rxSigFreq))

#     rxSigFreq = tf.cast(tf.round(rxSigFreq*2**13), tf.int16)
#     rxSigFreq = tf.reshape(rxSigFreq,[8,-1])
    
#     for rxIdx in range(8):
#         file_path = os.path.join(dir, f'rx_ant_{rxIdx}.bin')
#         with open(file_path, 'wb') as file:
#             file.write(rxSigFreq[rxIdx])
#     return rxSigFreq



# def setRef(inBits, dir):
#     filename = os.path.join(dir, 'ref.txt')

#     if inBits == None: 
#         with open(filename, 'w') as file:
#             file.write(f'##----------------------------------------------------------------------------\n')
#         return 
    
#     with open(filename, 'w') as file:
#         file.write(f'##----------------------------------------------------------------------------\n')
#         fn = -1
#         for n,payload in enumerate(inBits):
#             # payload = inBits[:, i]
#             if n in [4,5,14,15]:
#                 fn = fn + 1
#                 tbSize = payload.shape[-1]//8
#                 file.write(f'#type[PUSCH] fn[{fn}] slot[{n}] sym[0] carrier[0] chanId[0] len[{tbSize}]\n')
#                 file.write(f'\t  #ta[0] cqi[0.0] stat[1]\n')
#                 file.write(f'\t  #data[\n')
#                 Q = tbSize//64
#                 payload = tf.math.reduce_sum(tf.reshape(payload,[-1, 8]) * tf.constant([[128, 64, 32, 16, 8, 4, 2, 1]], dtype=tf.uint8), axis=1)
#                 # print(payload.shape, Q, r)
#                 for q in range(Q):
#                     file.write('\t        ')
#                     for r in range(64):
#                         file.write(f'{payload[64*q + r]:3d}, ')
#                     file.write('\n')
#                 file.write('\t        ')
#                 for r in range(tbSize%64-1):
#                     file.write(f'{payload[64*Q + r]:3d}, ')
#                 file.write(f'{payload[-1]:3d}\n')
#                 file.write(f'\t       ]\n')
#                 file.write(f'------------------------------------------------------------------------------\n')
#             else: 
#                 file.write(f'##----------------------------------------------------------------------------\n')

